In [1]:
from access_parser import AccessParser
import pandas as pd

DB = "../final_speciate_5.4_2025-09-08/SPECIATE 5.4 2025-09-08.accdb"
db = AccessParser(DB)
for t in db.catalog:
    if t.startswith("MSys"):
        continue
    tbl = db.parse_table(t)
    print(t, len(next(iter(tbl.values()))) if tbl else 0, list(tbl.keys()))

ConcatenatedResult 6654 ['PROFILE_CODE', 'PROFILE_NAME', 'VERSION', 'PROFILE_TYPE', 'DATA_ORIGIN', 'REF_PRIMARY', 'KEYWORD', 'REF_DOCUMENTS', 'REF_DESCRIPTION']
f_8A24C02D37834628BEFB20C3E1EBA07E_Data 1 ['FileFlags', 'FileTimeStamp', '_Data', 'FileData', 'FileName', 'FileType', 'FileURL', 'MSysResources_Data']
List of SVOC Splitting Factors 236 ['SPECIES_ID', 'PM Mass Fraction', 'NAME', 'Pollutant Code']
MNEMONIC 3939 ['ID', 'PROFILE_TYPE', 'PROFILE_CODE', 'DRI_PNUMBER', 'MNEMONIC']
Oxide Forms 34 ['Species', 'Oxide Form 1', 'Oxide Form 2', 'Oxide Form 3', 'Oxygen/Metal Ratio']
PROFILE_REFERENCE_CROSSWALK 9132 ['PROFILE_CODE', 'REF_Code']
PROFILES 6897 ['QSCORE', 'PROFILE_DATE', 'TOTAL', 'STANDARD', 'INCLUDES_INORGANIC GAS', 'JUDGEMENT_RATING', 'VINTAGE_RATING', 'DATA_QUANTITY_RATING', 'LOWER_SIZE', 'UPPER_SIZE', 'TOG_to_VOC RATIO', 'TEMP_SAMPLE_C', 'RH_SAMPLE', 'PARTICLE_LOADING_ug_per_m3', 'ORGANIC_LOADING_ug_per_m3', 'MASTER_POLLUTANT_EMISSION_RATE', 'ORGANIC_MATTER_to_ORGANIC_CARBO

In [2]:
tabs = {t: pd.DataFrame(db.parse_table(t)) for t in
        ["SPECIES", "PROFILES", "SPECIES_PROPERTIES", "tblSpeciesAndConcatSynonyms"]}

species = tabs["SPECIES"]
profiles = tabs["PROFILES"][["PROFILE_CODE", "PROFILE_NAME", "PROFILE_TYPE", "MASTER_POLLUTANT", "VERSION",
                             "CATEGORY_LEVEL_1_Generation_Mechanism", "CATEGORY_LEVEL_2_Sector_Equipment",
                             "CATEGORY_LEVEL_3_ Fuel_Product", "Keywords"]]
props = tabs["SPECIES_PROPERTIES"][["SPECIES_ID", "CAS", "CAS no hyphen", "DSSTox_ID", "SPECIES_NAME",
                                    "Molecular Formula", "Smiles Notation"]]
syn = tabs["tblSpeciesAndConcatSynonyms"][["SPECIES_ID", "SPECIES_NAMES"]]

# Check join keys are unique on the lookup side
for name, d, k in [("profiles", profiles, "PROFILE_CODE"), ("props", props, "SPECIES_ID"), ("syn", syn, "SPECIES_ID")]:
    print(name, "dup keys:", d[k].duplicated().sum())

flat = (
    species
    .merge(profiles, on="PROFILE_CODE", how="left", validate="m:1")
    .merge(props, on="SPECIES_ID", how="left", validate="m:1")
    .merge(syn[["SPECIES_ID", "SPECIES_NAMES"]], on="SPECIES_ID", how="left", validate="m:1")
    .rename(columns={"SPECIES_NAME": "Name"})
)
print(len(species), len(flat))
print(flat[["PROFILE_NAME", "Name", "CAS", "CAS no hyphen", "SPECIES_NAMES", "WEIGHT_PERCENT"]].isna().mean())
flat.head()

profiles dup keys: 0
props dup keys: 0
syn dup keys: 0
348268 348268
PROFILE_NAME      0.000000
Name              0.000000
CAS               0.000014
CAS no hyphen     0.000014
SPECIES_NAMES     0.007661
WEIGHT_PERCENT    0.017386
dtype: float64


,SPECIES_ID,WEIGHT_PERCENT,UNCERTAINTY_PERCENT,SPECIES_EMISSION_RATE,PROFILE_CODE,INCLUDE_IN_SUM,UNCERTAINTY_METHOD,ANALYTICAL_METHOD,PHASE,SPECIES_EMISSION_RATE_UNIT,...,CATEGORY_LEVEL_2_Sector_Equipment,CATEGORY_LEVEL_3_ Fuel_Product,Keywords,CAS,CAS no hyphen,DSSTox_ID,Name,Molecular Formula,Smiles Notation,SPECIES_NAMES
0,4,0.43,-99.0,NaN,0000,Yes,N/A,Not Available,GAS,None,...,Miscellaneous,Miscellaneous,OVERALL AVERAGE,71-55-6,71556,DTXSID0021381,"1,1,1-trichloroethane",C2H3Cl3,CC(Cl)(Cl)Cl,"1,1,1-trichloroethane || Methylchloroform || E..."
1,7,0.35,-99.0,NaN,0000,Yes,N/A,Not Available,GAS,None,...,Miscellaneous,Miscellaneous,OVERALL AVERAGE,79-00-5,79005,DTXSID5021380,"1,1,2-trichloroethane",C2H3Cl3,ClCC(Cl)Cl,"1,1,2-trichloroethane || Ethane, 1,1,2-trichlo..."
2,12,0.02,-99.0,NaN,0000,Yes,N/A,Not Available,GAS,None,...,Miscellaneous,Miscellaneous,OVERALL AVERAGE,3073-66-3,3073663,DTXSID60858746,"1,1,3-trimethylcyclohexane",C9H18,CC1CCCC(C)(C)C1,"1,1,3-trimethylcyclohexane || Cyclohexane, 1,1..."
3,30,0.01,-99.0,NaN,0000,Yes,N/A,Not Available,GAS,None,...,Miscellaneous,Miscellaneous,OVERALL AVERAGE,95-63-6,95636,DTXSID6021402,"1,2,4-trimethylbenzene",C9H12,CC1=CC(C)=C(C)C=C1,"1,2,4-trimethylbenzene || Benzene, 1,2,4-trime..."
4,44,0.03,-99.0,NaN,0000,Yes,N/A,Not Available,GAS,None,...,Miscellaneous,Miscellaneous,OVERALL AVERAGE,108-67-8,108678,DTXSID6026797,"1,3,5-trimethylbenzene",C9H12,CC1=CC(C)=CC(C)=C1,"1,3,5-trimethylbenzene || Benzene, 1,3,5-trime..."


In [3]:
OUT = "SPECIATE_Profiles_All_02.csv"
flat.to_csv(OUT, index=False, encoding="utf-8")
print(flat.shape, flat["CAS no hyphen"].map(type).value_counts().to_dict())

(348268, 25) {<class 'str'>: 348263, <class 'NoneType'>: 5}
